# 04. Recording-time confound on D2 (E6)

Leak vs. no-leak from recording time only (audio not used).

- Collection protocol (dataset description): leak recorded when reported; no-leak recorded after repair at a previously visited site
- Unit: file. 160 public files = 140 WAV + 20 MEMS XLSX; 140 WAV = 118 timed + 22 untimed
- Intervals: Wilson (file), bootstrap (logger)
- Output for Figure 4: `results/D2_recording_times.csv` (n = 118)
- Acoustic vs. clock comparison: notebook 07

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation as ev, sfb, reporting as rp
cfg.apply_mpl_style(); cfg.banner()
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import LeaveOneOut, GroupKFold, cross_val_predict

M2 = io_utils.scan_d2(audio_info=True)
print(f"D2 files: {len(M2)}   loggers/sites: {M2.logger.nunique()}")
print(M2.groupby(["sensor", "folder", "ext"]).size().to_string())

LeakAudit | root=C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit | mode=FULL | seeds=10 | bootstrap=2000
D2 files: 160   loggers/sites: 50
sensor               folder           ext  
Hydrophones          Leak             .wav     40
                     No-Leak          .wav     40
MEMS Accelerometers  Leak             .xlsx    10
                     No-Leak          .xlsx    10
Noise Loggers        Leak-Metal       .wav     15
                     Leak-NonMetal    .wav     15
                     NoLeak-Metal     .wav     15
                     NoLeak-NonMetal  .wav     15


## 1. Population accounting

In [2]:
pop = io_utils.d2_population(M2)
pop.to_csv(cfg.RESULTS / "D2_population.csv", index=False)
pd.set_option("display.max_colwidth", 70)
print(pop.to_string(index=False))
for _, r in pop.iterrows():
    rp.record(f"D2.pop.{r.population}.n", r.n, "D2_population.csv", "R1-M2,R2-5",
              note=f"leak={r.n_leak}, noleak={r.n_noleak}, majority={r.majority_rate:.4f}")

wav = M2[(M2.ext == ".wav") & M2.label.isin(["leak", "noleak"])].reset_index(drop=True)
print("\nnaming convention x label (WAV files):")
print(pd.crosstab([wav.sensor, wav.naming], [wav.label, wav.timed]).to_string())

             population unit   n  n_leak  n_noleak  majority_rate  n_loggers                                       sensors                                                            used_for
       all public files file 160      80        80       0.500000         50 Hydrophones|MEMS Accelerometers|Noise Loggers                                                      inventory only
             MEMS .xlsx file  20      10        10       0.500000          0                           MEMS Accelerometers accelerometer spreadsheets; not audio; excluded from every analysis
              WAV files file 140      70        70       0.500000         50                     Hydrophones|Noise Loggers                         pretrained-embedding analysis (Section 4.5)
    WAV, parseable time file 118      48        70       0.593220         32                     Hydrophones|Noise Loggers                        recording-time stump (Section 4.3), Figure 4
 WAV, no parseable time file  22      22     

## 2. Untimed files

- 22 untimed WAV: all leak, all noise-logger files named `NNN_LOGGER-YYYYMMDD`
- Timed subset: 48 leak / 70 no-leak → majority rate 0.593
- Naming convention as a signal-free predictor: notebook 10

In [ ]:
untimed = wav[~wav.timed]
summ = dict(n_untimed_wav=int(len(untimed)),
            untimed_labels=untimed.label.value_counts().to_dict(),
            untimed_naming=untimed.naming.value_counts().to_dict(),
            untimed_sensor=untimed.sensor.value_counts().to_dict(),
            n_mems_xlsx=int((M2.ext == ".xlsx").sum()),
            total_without_time=int(len(untimed) + (M2.ext == ".xlsx").sum()))
if "sr" in wav.columns:
    fmt = wav.groupby(["timed", "label"]).agg(sr=("sr", lambda s: sorted(set(s.dropna()))),
                                              dur_s=("duration_s", lambda s: [round(float(np.nanmin(s)), 1),
                                                                              round(float(np.nanmax(s)), 1)]))
    print(fmt.to_string())
    summ["format_by_timed_label"] = fmt.reset_index().astype(str).to_dict(orient="records")
for k, v in summ.items():
    print(f"  {k:24s} {v}")
la.save_json(summ, "D2_untimed_files.json")

both = wav.groupby("logger").label.nunique()
matched = both[both > 1].index.tolist()
print(f"\nloggers/sites with both classes: {len(matched)} {matched}")
la.save_json(dict(loggers_with_both_classes=matched,
                  n_loggers=int(wav.logger.nunique())), "D2_matched_sites.json")

                        sr         dur_s
timed label                             
False leak          [4096]  [10.0, 20.0]
True  leak    [4096, 8092]  [10.0, 10.0]
      noleak  [4096, 8092]  [10.0, 20.0]
  n_untimed_wav            22
  untimed_labels           {'leak': 22}
  untimed_naming           {'idx_logger_date': 22}
  untimed_sensor           {'Noise Loggers': 22}
  n_mems_xlsx              20
  total_without_time       42
  format_by_timed_label    [{'timed': 'False', 'label': 'leak', 'sr': '[4096]', 'dur_s': '[10.0, 20.0]'}, {'timed': 'True', 'label': 'leak', 'sr': '[4096, 8092]', 'dur_s': '[10.0, 10.0]'}, {'timed': 'True', 'label': 'noleak', 'sr': '[4096, 8092]', 'dur_s': '[10.0, 20.0]'}]

loggers/sites with both classes: 0 []


WindowsPath('C:/Users/ecredible/Desktop/LeakAudit_revision1/LeakAudit/results/D2_matched_sites.json')

## 3. Recording time by class (Figure 4 data)

In [4]:
sub = wav[wav.timed].reset_index(drop=True)
print(pd.crosstab(sub.label, sub.hhmm).to_string())
print(f"\nmean hour  leak = {sub[sub.label=='leak'].hour.mean():.2f} h   "
      f"noleak = {sub[sub.label=='noleak'].hour.mean():.2f} h")
early = sub[(sub.hour >= 3.5) & (sub.hour <= 4.5)]
print(f"no-leak files recorded 03:30-04:30: {int((early.label=='noleak').sum())} of "
      f"{int((sub.label=='noleak').sum())}")
sub[["file_id", "sensor", "logger", "label", "hhmm", "hour"]].to_csv(
    cfg.RESULTS / "D2_recording_times.csv", index=False)
fmt_hm = lambda h: f"{int(h):02d}:{int(round((h % 1) * 60)):02d}"
for lab in ["leak", "noleak"]:
    rp.record(f"D2.mean_hour.{lab}", sub[sub.label == lab].hour.mean(), "D2_recording_times.csv",
              "R1-M2,R2-5", note=fmt_hm(sub[sub.label == lab].hour.mean()))

hhmm    0330  0345  0400  0415  0430  0630  0730  0830  0930  1030  1145  1248  1445  1538  1800  1900  2000  2100  2200
label                                                                                                                   
leak      14    14    10    10     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0
noleak     4     4     5     5     4     2     2     2     2     2    24     1     2     1     2     2     2     2     2

mean hour  leak = 3.83 h   noleak = 10.19 h
no-leak files recorded 03:30-04:30: 22 of 70


## 4. Signal-free baseline: depth-1 stump on hour

- Validation: leave-one-out, logger-independent (GroupKFold by logger)
- Intervals: Wilson (file), cluster bootstrap (logger)

In [5]:
def stump_report(frame, name):
    Xh, yh, lg = frame[["hour"]].to_numpy(), frame.label.to_numpy(), frame.logger.to_numpy()
    stump = DecisionTreeClassifier(max_depth=1, random_state=0)
    base = sfb.majority_baseline(yh)
    maj = pd.Series(yh).value_counts().idxmax()
    out = {}
    for proto, cv, groups in [("LOO", LeaveOneOut(), None),
                              ("logger_indep", GroupKFold(min(5, len(set(lg)))), lg)]:
        pred = cross_val_predict(stump, Xh, yh, cv=cv, groups=groups)
        ok = pred == yh
        k, n = int(ok.sum()), len(ok)
        w = sfb.wilson_ci(k, n)
        cb = sfb.cluster_bootstrap_acc(ok, lg, n_boot=cfg.N_BOOT, base_correct=(yh == maj))
        thr = DecisionTreeClassifier(max_depth=1).fit(Xh, yh).tree_.threshold[0]
        out[proto] = dict(population=name, protocol=proto, n=n, acc=k / n,
                          wilson_lo=w[0], wilson_hi=w[1],
                          logger_boot_lo=cb["lo"], logger_boot_hi=cb["hi"],
                          n_loggers=cb["n_clusters"], baseline=base, excess=k / n - base,
                          excess_lo=cb["excess_lo"], excess_hi=cb["excess_hi"],
                          full_data_threshold_h=float(thr))
    return out

res = stump_report(sub, "WAV, parseable time (n=118)")
tab = pd.DataFrame(res.values())
print(tab.round(4).to_string(index=False))

                 population     protocol   n    acc  wilson_lo  wilson_hi  logger_boot_lo  logger_boot_hi  n_loggers  baseline  excess  excess_lo  excess_hi  full_data_threshold_h
WAV, parseable time (n=118)          LOO 118 0.8475     0.7717     0.9013          0.7187          0.9608         32    0.5932  0.2542        0.0     0.5138                  4.375
WAV, parseable time (n=118) logger_indep 118 0.8475     0.7717     0.9013          0.7187          0.9608         32    0.5932  0.2542        0.0     0.5138                  4.375


## 5. Early-morning window (03:30–04:30)

- Both classes present: 48 leak, 22 no-leak
- No logger with both classes → site-matched comparison not possible

In [ ]:
res_e = stump_report(early.reset_index(drop=True), "WAV, 03:30-04:30 (n=70)")
tab_e = pd.DataFrame(res_e.values())
print(tab_e.round(4).to_string(index=False))

e6 = pd.concat([tab, tab_e], ignore_index=True)
e6.to_csv(cfg.RESULTS / "E6_time_confound_D2.csv", index=False)
r = res["logger_indep"]; r0 = res["LOO"]
la.save_json(dict(n=r["n"], acc_hour_loo=r0["acc"], acc_hour_grouped=r["acc"],
                  baseline=r["baseline"],
                  loo_wilson=[r0["wilson_lo"], r0["wilson_hi"]],
                  loo_logger_boot=[r0["logger_boot_lo"], r0["logger_boot_hi"]],
                  grouped_logger_boot=[r["logger_boot_lo"], r["logger_boot_hi"]],
                  mean_hour_leak=float(sub[sub.label == 'leak'].hour.mean()),
                  mean_hour_noleak=float(sub[sub.label == 'noleak'].hour.mean()),
                  early_window=res_e), "E6_time_confound_D2.json")
for proto, rr in res.items():
    rp.record(f"D2.clock_stump.{proto}.acc", rr["acc"], "E6_time_confound_D2.csv",
              "R1-M7,R2-11", lo=rr["logger_boot_lo"], hi=rr["logger_boot_hi"],
              note="interval: logger-cluster bootstrap; n=118 files")
    rp.record(f"D2.clock_stump.{proto}.excess", rr["excess"], "E6_time_confound_D2.csv",
              "R1-M7", lo=rr["excess_lo"], hi=rr["excess_hi"])
for proto, rr in res_e.items():
    rp.record(f"D2.early_window.clock_stump.{proto}.acc", rr["acc"], "E6_time_confound_D2.csv",
              "R2-6", note=f"baseline={rr['baseline']:.4f}, n={rr['n']}")

             population     protocol  n    acc  wilson_lo  wilson_hi  logger_boot_lo  logger_boot_hi  n_loggers  baseline  excess  excess_lo  excess_hi  full_data_threshold_h
WAV, 03:30-04:30 (n=70)          LOO 70 0.7429     0.6297     0.8307          0.5526          0.9178         19    0.6857  0.0571     0.0149     0.1026                  4.375
WAV, 03:30-04:30 (n=70) logger_indep 70 0.7429     0.6297     0.8307          0.5526          0.9178         19    0.6857  0.0571     0.0149     0.1026                  4.375

-> the gap above the baseline is the confound the audio cannot be separated from.
